# Advanced Retrieval Strategies: Implementing Multi-Query RAG

Retrieval-Augmented Generation (RAG) has become the backbone of enterprise AI applications, allowing LLMs to ground their responses in proprietary knowledge bases. However, standard RAG pipelines often struggle when user queries are complex, ambiguous, or require synthesizing information from multiple angles. A single query might only activate a narrow set of relevant documents, leading to incomplete or inaccurate answers.

This notebook introduces the concept of **Multi-Query Retrieval**, an advanced technique that significantly boosts the robustness and recall of RAG systems. Instead of relying solely on the initial user prompt for retrieval, we employ an LLM to first generate several related sub-queries (or "query expansions"). These multiple queries are then used independently against the vector store, allowing us to gather a diverse set of relevant documents that cover all facets of the original request.

By mastering multi-query strategies, developers can build highly resilient and comprehensive retrieval pipelines. This pattern is foundational for advanced state machine orchestration using frameworks like LangGraph, where the retrieval step itself becomes a complex, multi-step agentic process—first querying, then aggregating results, and finally passing the enriched context to the generator. By the end of this notebook, you will be equipped to design sophisticated retrievers that move beyond simple similarity search toward deep semantic understanding.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Understand Limitations:** Identify scenarios where single-query retrieval fails or provides insufficient context.
*   **Implement Multi-Query Generation:** Utilize LLMs (e.g., `ChatOpenAI`) to programmatically generate multiple related sub-queries from a single user prompt.
*   **Execute Advanced Retrieval:** Integrate the generated queries into a vector store (`InMemoryVectorStore`) to retrieve a comprehensive set of documents, maximizing context coverage.
*   **Design Robust Pipelines:** Conceptualize how multi-query retrieval can be integrated as an initial, complex state within a larger LangGraph workflow.


### Setup and Imports

This cell imports all necessary libraries, types, and core components from LangChain and related packages. It sets up the foundational tools for building a sophisticated RAG system, including document handling (`Document`), retrieval mechanisms (`BaseRetriever`, `InMemoryVectorStore`), prompt templating (`ChatPromptTemplate`), embedding models (`OpenAIEmbeddings`), LLMs (`ChatOpenAI`), text splitting utilities (`RecursiveCharacterTextSplitter`), and structured data validation (`BaseModel`).


In [ ]:
from dotenv import load_dotenv # Used to load environment variables (e.g., API keys)
from typing import Any # Standard Python type hinting for generic types
from langchain_core.documents import Document # Core class for representing documents in LangChain
from langchain_core.retrievers import BaseRetriever # Abstract base class for all retrievers
from langchain_core.vectorstores import InMemoryVectorStore # A simple, in-memory vector store implementation
from langchain_core.prompts import ChatPromptTemplate # Tool for defining structured chat prompts
from langchain_openai import OpenAIEmbeddings, ChatOpenAI # Specific implementations of embeddings and LLMs using OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter # Utility to split large texts into smaller chunks
from pydantic import BaseModel, Field # Used for defining structured output schemas (Pydantic models)


In [ ]:
load_dotenv()

True

### Initialization of Core Components

This cell initializes the essential components for our RAG pipeline: the embedding model and the Large Language Model (LLM). `OpenAIEmbeddings` converts text into numerical vectors, while `ChatOpenAI` provides the generative AI capabilities needed for reasoning and response generation.


In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small") # Initializes the embedding model used to convert text chunks into high-dimensional vector representations.
llm = ChatOpenAI(model="gpt-5-mini", temperature=0.3) # Initializes the chat-based LLM, which will be used for generating answers and reasoning, with a controlled randomness (temperature).



### Data Preparation and Loading

This cell initializes a list of `Document` objects, each containing specialized knowledge in different scientific/tech domains (e.g., biotechnology, cybersecurity). It simulates loading diverse, structured data into the system's memory using the LangChain `Document` class, which is essential for subsequent Retrieval-Augmented Generation (RAG) steps.


In [4]:
docs = [
    Document(
        page_content=(
            "Biotechnology companies are developing novel protein-based therapies that target specific "
            "disease pathways with unprecedented precision. Synthetic biology techniques allow scientists "
            "to engineer microorganisms that produce pharmaceutical compounds at industrial scale. "
            "Bioreactor technologies have dramatically reduced the cost of producing monoclonal antibodies, "
            "making treatments for autoimmune diseases and cancers more accessible. Microbiome research is "
            "revealing how manipulating gut bacteria can influence everything from mental health to "
            "metabolic disorders."
        ),
        metadata={"topic": "biotechnology"},
    ),
    Document(
        page_content=(
            "Zero-trust architecture has become the gold standard for enterprise network security, "
            "requiring continuous verification rather than relying on perimeter defenses. Machine learning "
            "models now detect anomalous network behavior in real time, reducing the window between "
            "intrusion and detection from months to minutes. Ransomware attacks on critical infrastructure "
            "have forced governments to establish mandatory incident reporting requirements for healthcare "
            "and energy sectors. Post-quantum cryptography standards are being finalized to protect "
            "sensitive data against future quantum computing threats."
        ),
        metadata={"topic": "cybersecurity"},
    ),
    Document(
        page_content=(
            "Brain-computer interfaces are enabling paralyzed patients to control prosthetic limbs and "
            "communicate using only their neural signals. Optogenetics allows researchers to activate or "
            "silence specific neuron populations with light, accelerating the understanding of neural "
            "circuit function and disease. Advanced neuroimaging techniques using fMRI and "
            "magnetoencephalography are mapping brain connectivity with millimeter precision, unlocking "
            "new treatments for depression and PTSD. Neurofeedback therapies are showing promise for "
            "cognitive rehabilitation following traumatic brain injuries."
        ),
        metadata={"topic": "neuroscience"},
    ),
    Document(
        page_content=(
            "Perovskite solar cells have achieved efficiency ratings exceeding 33%, surpassing traditional "
            "silicon panels and promising dramatically lower manufacturing costs. Grid-scale battery "
            "storage using iron-air and sodium-ion technologies is making renewable energy dispatchable "
            "around the clock without relying on rare earth metals. Offshore floating wind farms are "
            "expanding into deep-water regions previously inaccessible to fixed-foundation turbines, "
            "multiplying available wind energy capacity. Green hydrogen produced via electrolysis is "
            "emerging as a critical energy carrier for decarbonizing heavy industry and long-haul "
            "transport."
        ),
        metadata={"topic": "renewable_energy"},
    ),
    Document(
        page_content=(
            "Surgical robots equipped with haptic feedback allow surgeons to perform minimally invasive "
            "procedures with sub-millimeter precision, reducing patient recovery times significantly. "
            "Collaborative robots in manufacturing now work safely alongside humans using advanced "
            "computer vision and force sensing, without the need for physical barriers. Autonomous mobile "
            "robots are transforming warehouse logistics, optimizing pick-and-place operations and "
            "reducing fulfillment errors. Soft robots inspired by biological organisms are being developed "
            "for delicate tasks in agriculture, search-and-rescue, and medical drug delivery."
        ),
        metadata={"topic": "robotics"},
    ),
    Document(
        page_content=(
            "Base editing and prime editing technologies offer more precise alternatives to CRISPR-Cas9, "
            "enabling single-letter corrections to the genome without creating double-strand breaks. "
            "Gene therapy trials using adeno-associated virus vectors have achieved functional cures for "
            "hemophilia B and spinal muscular atrophy. Epigenome editing tools allow researchers to "
            "switch genes on or off without altering the underlying DNA sequence, opening new avenues "
            "for treating complex diseases. Polygenic risk scoring combined with germline analysis is "
            "enabling predictive medicine that identifies disease susceptibility decades before symptoms "
            "appear."
        ),
        metadata={"topic": "genetic_engineering"},
    ),
]

print(f"Created {len(docs)} documents") # Print the total count of documents created for verification



Created 6 documents


### Document Chunking (Text Splitting)

This step uses `RecursiveCharacterTextSplitter` to break down large documents (`docs`) into smaller, manageable chunks. This process is crucial for RAG because embedding models have token limits and retrieving small, focused chunks improves the relevance of context provided to the LLM.


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

docs = splitter.split_documents(docs) # Splits the original documents into a list of smaller chunks (documents)


### Vectorstore Initialization and Retriever Setup

This cell first initializes an in-memory vector store from the loaded documents (`docs`) using specified embeddings. It then converts this vector store into a retriever object, which is the core component used to fetch relevant document chunks based on similarity search (k=3).


In [ ]:
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings) # Initializes an in-memory vector store using the loaded documents and specified embeddings.

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) # Converts the vectorstore into a retriever object, configured to retrieve the top 3 most relevant chunks (k=3).


### Code Explanation

This cell implements a custom `CustomMultiQueryRetriever` that enhances standard retrieval by generating multiple alternative phrasings of the user's query using an LLM. It then executes the base retriever for each generated query and deduplicates the resulting documents, improving recall by covering diverse semantic angles.


In [ ]:
# QueriesSchema is the structured output the LLM produces — a list of alternative questions
class QueriesSchema(BaseModel):
    queries: list[str] = Field(description="List of 3 alternative versions of the question")


prompt = ChatPromptTemplate.from_template(
    "You are an AI language model assistant. Your task is to generate 3 different versions of "
    "the given user question to retrieve relevant documents from a vector database. "
    "By generating multiple perspectives on the user question, your goal is to help the user "
    "overcome some of the limitations of distance-based similarity search. "
    "Provide these alternative questions separated by newlines.\n\n"
    "Original question: {question}"
)

llm_structured_output = llm.with_structured_output(QueriesSchema)

# with_structured_output binds QueriesSchema so the chain always returns a QueriesSchema instance
query_chain = prompt | llm_structured_output


class CustomMultiQueryRetriever(BaseRetriever):
    """Retriever that generates multiple query perspectives via an LLM and deduplicates results."""

    base_retriever: BaseRetriever
    query_chain: Any  # prompt | llm.with_structured_output(QueriesSchema)

    def _generate_queries(self, query: str) -> list[str]:
        # Invokes the LLM chain to generate a structured list of alternative queries
        result: QueriesSchema = self.query_chain.invoke({"question": query})
        return result.queries

    def _unique_documents(self, documents: list[Document]) -> list[Document]:
        # Deduplicate the retrieved documents based on their page content.
        # This ensures that if multiple queries retrieve the same document chunk, it is only counted once.
        seen: set[str] = set()
        unique: list[Document] = []
        for doc in documents:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                unique.append(doc)
        return unique

    def _get_relevant_documents(self, query: str) -> list[Document]:
        # Step 1: generate alternative query phrasings using the LLM.
        queries = self._generate_queries(query)
        # Step 2: retrieve docs for each alternative query using the base retriever.
        all_docs: list[Document] = []
        for q in queries:
            all_docs.extend(self.base_retriever.invoke(q))
        # Step 3: deduplicate all collected documents and return the unique set.
        return self._unique_documents(all_docs)


This cell initializes and uses the `CustomMultiQueryRetriever`. It first leverages the internal `query_chain` to generate several alternative, optimized queries for the original user question. Then, it executes the retriever using these generated queries to fetch a comprehensive set of relevant documents.


In [ ]:
retriever = CustomMultiQueryRetriever(base_retriever=base_retriever, query_chain=query_chain)

# Define the main user query.
query = "How are modern technologies improving human health?"

# Peek at the alternative queries before seeing results. This step uses the query chain to generate and print all optimized search terms.
parsed = query_chain.invoke({"question": query})

print("Generated alternative queries:")
for q in parsed.queries:
    print(f"  - {q}")
print()

# Execute the retriever using the multi-query logic, fetching documents based on all generated queries.
results = retriever.invoke(query)
print(f"Retrieved {len(results)} unique documents:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} [{doc.metadata.get('topic')}] ---")
    print(doc.page_content)
    print()



Generated alternative queries:
  - How are recent technologies—such as AI, genomics, telemedicine, wearables, and robotics—improving diagnosis, treatment, prevention, and patient outcomes?
  - What empirical evidence shows that digital health tools (machine learning, remote monitoring, mobile health apps) and precision medicine improve population health and healthcare system efficiency?
  - Which modern medical technologies (gene editing, personalized therapies, advanced diagnostics, remote monitoring) have demonstrated measurable benefits in clinical outcomes, access to care, or cost reduction?

Retrieved 4 unique documents:

--- Result 1 [biotechnology] ---
Biotechnology companies are developing novel protein-based therapies that target specific disease pathways with unprecedented precision. Synthetic biology techniques allow scientists to engineer microorganisms that produce pharmaceutical compounds at industrial scale. Bioreactor technologies have dramatically reduced the cost of p